|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 6:</h2>|<h1>The Server<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: break it on purpose<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/'course/Part6_TheServer/3_incidents'))

import math, time
import torch
import lab
import asyncio, random, statistics
from transformers import AutoTokenizer

In the incident file you went from a symptom to a cause. Here you go the other
way. You put one fault into a working event loop, stream, proxy or metric, and
you watch what it does.

The routine for each exercise is the same:

1. Read the fault.
2. **Write your prediction in the cell.** Answer the four questions.
3. Run the cell.
4. Write down where your prediction was wrong. This line is the one that
   teaches you.

The four questions:

- **Crash?** Does it raise an error, or does it run?
- **When?** Which stream, which moment, which load?
- **What?** What does the wrong result look like: a stall, a leak, a number
  that lies?
- **Which guard?** Which check would catch it?

`lab.FakeEngine` runs on the event loop like the engine of stage 15, with no
GPU: each step sleeps for the step time, then gives one token to each running
request. `lab.client` consumes a stream and records when each token arrives.
The notebook runs real `asyncio` code, so the stalls are real stalls.

This notebook needs no GPU.

In [ ]:
### run this cell

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B')
DOCUMENT = 'The committee met on Tuesday to review the budget, the hiring plan and the new office lease. ' * 5000
print('the document has', len(tokenizer.encode(DOCUMENT)), 'tokens')

# Exercise 1: tokenize in the event loop

Eight streams run on the fake engine, with a step of 22 ms. After one second,
a request with a long document arrives. The handler tokenizes the document
with a plain call, in the event loop. Then run the same with the call in a
worker thread.

This is Ticket 1 of the incident file. Predict the worst gap between two
tokens of each stream, in each case.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
async def run(offload):
  engine = lab.FakeEngine(slots=16, step_ms=22)
  engine.start()
  record = {}
  streams = [asyncio.create_task(lab.client(engine, i, 120, record)) for i in range(8)]
  await asyncio.sleep(1.0)
  start = time.perf_counter()
  if offload:
    ids = await asyncio.get_running_loop().run_in_executor(None, tokenizer.encode, DOCUMENT)
  else:
    ids = tokenizer.encode(DOCUMENT)                                    # THE FAULT: blocks the loop
  took = time.perf_counter() - start
  await asyncio.gather(*streams)
  await engine.stop()
  worst = [max(lab.gaps(times)) * 1000 for times in record.values()]
  print(f'{"worker thread" if offload else "event loop   "}: tokenize {took * 1000:4.0f} ms, '
        f'the worst gap of each stream {min(worst):4.0f} to {max(worst):4.0f} ms')

await run(offload=False)
await run(offload=True)

# Exercise 2: the user presses stop

100 requests on an engine with 32 slots. 70% of the answers end after 40
tokens. 30% ramble to 400 tokens, and their users press stop after 5 tokens.
The stream handler closes, and nobody tells the engine. Then run the same
with `engine.abort` when the stream closes.

This is Ticket 2. Predict the ratio of the delivered tokens to the generated
tokens, and how long the engine stays busy.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
async def run(abort):
  engine = lab.FakeEngine(slots=32, step_ms=2)
  engine.start()
  rng = random.Random(0)
  record, clients = {}, []
  start = time.perf_counter()
  for rid in range(100):
    rambles = rng.random() < 0.3
    clients.append(lab.client(engine, rid, 400 if rambles else 40, record,
                              stop_after=5 if rambles else None,
                              on_close=engine.abort if abort else None))   # THE FAULT when None
  await asyncio.gather(*clients)
  users_done = time.perf_counter() - start
  while engine.running or engine.waiting:
    await asyncio.sleep(0.01)
  engine_done = time.perf_counter() - start
  await engine.stop()
  print(f'abort {str(abort):5s}: delivered {engine.delivered} of {engine.generated} generated '
        f'({engine.delivered / engine.generated:.0%}); the users were done after {users_done:.2f} s, '
        f'the engine after {engine_done:.2f} s')

await run(abort=False)
await run(abort=True)

# Exercise 3: start the clock at admission

40 requests arrive at once on an engine with 8 slots and a step of 10 ms.
Each answer has 50 tokens. Measure the time to the first token from the
arrival, as the user sees it, and from the admission, as the code of Ticket 3
does.

Predict both medians.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
engine = lab.FakeEngine(slots=8, step_ms=10)
engine.start()
record, arrival = {}, time.perf_counter()
await asyncio.gather(*[lab.client(engine, rid, 50, record) for rid in range(40)])
await engine.stop()
from_arrival = sorted(times[0] for times in record.values())
from_admission = sorted(arrival + record[rid][0] - engine.admitted_at[rid] for rid in record)   # THE FAULT
print(f'TTFT from the arrival  : median {statistics.median(from_arrival) * 1000:5.0f} ms, max {from_arrival[-1] * 1000:5.0f} ms')
print(f'TTFT from the admission: median {statistics.median(from_admission) * 1000:5.0f} ms, max {from_admission[-1] * 1000:5.0f} ms')

# Exercise 4: the average of the p99s

Ten pods. Nine are healthy. One has a GPU that throttles, and its requests
take ten times longer. Each pod serves 10% of the requests. The dashboard
shows the average of the ten p99 values.

This is Ticket 4. Predict the dashboard p99 and the true p99 of all the
requests.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
rng = random.Random(0)
pods = []
for pod in range(10):
  scale = 10 if pod == 9 else 1
  pods.append([scale * rng.lognormvariate(math.log(0.12), 0.25) for _ in range(10_000)])

def p99(values):
  return sorted(values)[int(0.99 * (len(values) - 1))]

dashboard = sum(p99(latencies) for latencies in pods) / len(pods)          # THE FAULT
everything = [x for latencies in pods for x in latencies]
print('p99 of each pod:', [round(p99(latencies) * 1000) for latencies in pods], 'ms')
print(f'the dashboard: {dashboard * 1000:.0f} ms    the true p99: {p99(everything) * 1000:.0f} ms')
print(f'the p90 of the bad pod: {sorted(pods[9])[int(0.9 * 9999)] * 1000:.0f} ms')

# Exercise 5: a proxy that buffers

The engine sends an answer of 400 tokens as Server-Sent Events, one event of
about 60 bytes every 25 ms. A proxy between the server and the user collects
the bytes, and sends them on only when its buffer is full or the response
ends. Simulate buffers of 0 (no buffering), 4 KB and 32 KB.

This is Ticket 5. Predict the time to the first token and the time of the
whole answer that the user sees, for each buffer.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
events = [(i * 0.025 + 0.15, len(f'data: {{"id": "chunk-{i}", "text": " word{i}"}}\n\n')) for i in range(400)]
print(f'{len(events)} events, {sum(size for _, size in events):,} bytes')

def through_proxy(events, buffer_bytes):
  delivered, held = [], 0
  for t, size in events:
    held += size
    if held >= buffer_bytes:                                            # THE FAULT when the buffer is large
      delivered.append(t)
      held = 0
  if held:
    delivered.append(events[-1][0])
  return delivered

for buffer in (0, 4096, 32768):
  out = through_proxy(events, buffer)
  print(f'buffer {buffer:6,} B: first bytes at {out[0]:5.2f} s, the last at {out[-1]:5.2f} s, '
        f'{len(out)} deliveries')

# Exercise 6: a load test with no think time

A simulated engine with 64 slots and a step of 13 ms + 0.25 ms for each
running request. Each answer has 200 tokens. First, a load test: 50 users
that send the next request as soon as the answer ends. Then production: the
same 50 users, who read for 20 s on average between two requests.

This is Ticket 7. Predict the tokens/s of each, and the mean number of
running requests.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
for name, think in [('load test, no think time', 0.0), ('production, 20 s to read', 20.0)]:
  rate, running, duration = lab.simulate_users(users=50, think_s=think)
  print(f'{name:26s}: {rate:6.0f} tok/s, {running:4.1f} requests running on average, '
        f'{duration:4.1f} s for each request')
print('Little: running = users x duration / (duration + think)')

# Exercise 7: three mystery metrics

The module `mystery.py` holds three metric functions. Each takes a log: a
list of dicts with `arrival`, `first` (the first token), `finish`, `tokens`
(the output tokens) and `done` (False for a request that timed out). Each one
has one fault. **Do not open the file.**

- `tpot(log)`: the mean time for each output token after the first.
- `ttft_p99(log)`: the p99 of the time to the first token.
- `throughput(log)`: output tokens per second of wall time.

`lab` has no reference for these. You know the definitions, so you can build
a log where you know every answer.

The cell below runs them on a simple log: one request at a time, no queue,
nothing times out. All three look plausible.

For each function:

1. Build a small log where you can compute the true value by hand, and that
   reaches the fault.
2. Write your diagnosis: the fault, and the log that proved it.
3. Only then, open `mystery.py` and check.

A hint about the method: one fault needs a request that waited before its
first token. One fault needs requests that did not finish. One fault needs
requests that overlap in time.

In [ ]:
from mystery import tpot, ttft_p99, throughput

simple = []
for i in range(200):
  start = i * 2.55                                                      # back to back
  simple.append(dict(arrival=start, first=start + 0.05, finish=start + 2.55, tokens=101, done=True))
print(f'tpot {tpot(simple) * 1000:.1f} ms,  ttft_p99 {ttft_p99(simple) * 1000:.0f} ms,  '
      f'throughput {throughput(simple):.1f} tok/s')

**Your diagnosis**

- `tpot`: the fault, and the experiment that proves it:
- `ttft_p99`: the fault, and the experiment that proves it:
- `throughput`: the fault, and the experiment that proves it:

# Your fingerprint table

Fill in this table from what you saw, not from what you predicted.

| Fault | Crash? | When it shows | What it looks like | The guard |
|---|---|---|---|---|
| tokenize in the event loop | | | | |
| no abort when the user stops | | | | |
| the TTFT clock starts at admission | | | | |
| the average of the p99s | | | | |
| a proxy that buffers | | | | |
| a load test with no think time | | | | |